In [ ]:
import os
import cv2
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage import segmentation, morphology
from dataclasses import dataclass

# ---------------------------------------------------------
# 1. Configuration & Hyperparameters
# ---------------------------------------------------------
@dataclass
class Config:
    # ROI dimensions referencing standard CR-39 darkfield scans
    roi_x: int = 2188
    roi_y: int = 2244
    roi_w: int = 5072
    roi_h: int = 4960
    
    # Detector thresholds
    det_min_area: int = 20
    det_fixed_thresh: int = 1
    
    # Watershed parameters for splitting touching tracks
    ws_min_distance: int = 3
    
    # Neural Network & GMM hyperparameters
    patch_size: int = 48
    latent_dim: int = 32
    ae_epochs: int = 25
    ae_batch: int = 256
    ae_lr: float = 1e-3
    n_clusters: int = 4
    random_state: int = 42

CFG = Config()

# ---------------------------------------------------------
# 2. Convolutional Autoencoder Architecture
# ---------------------------------------------------------
class TrackAutoencoder(nn.Module):
    """
    Unsupervised feature extractor that learns spatial embeddings
    from morphological track candidate patches.
    """
    def __init__(self, latent_dim=32):
        super(TrackAutoencoder, self).__init__()
        
        # Encoder (reduces 48x48 -> latent_dim)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, latent_dim) 
        )
        
        # Decoder (reconstructs latent_dim -> 48x48)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 12 * 12),
            nn.ReLU(),
            nn.Unflatten(1, (32, 12, 12)),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

# ---------------------------------------------------------
# 3. Morphological Processing & Patch Extraction
# ---------------------------------------------------------
def isolate_track_patches(image_path: str, cfg: Config):
    """
    Loads a CR-39 track image, applies thresholding and watershed 
    segmentation to segregate touching objects, and extracts square patches.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        # Create dummy data for execution safety if file is unreadable
        print(f"Warning: {image_path} not found. Generating simulated data.")
        img = np.zeros((cfg.roi_y + cfg.roi_h, cfg.roi_x + cfg.roi_w), dtype=np.uint8)
        cv2.circle(img, (cfg.roi_x + 100, cfg.roi_y + 100), 15, 255, -1)
    
    # Crop the Region of Interest
    roi = img[cfg.roi_y : cfg.roi_y + cfg.roi_h, cfg.roi_x : cfg.roi_x + cfg.roi_w]
    if roi.size == 0:
        return [], []
    
    # 1st morphological pass: Intensity thresholding
    mask = roi >= cfg.det_fixed_thresh
    mask = morphology.remove_small_objects(mask, min_size=cfg.det_min_area)
    
    # 2nd morphological pass: Distance Transform & Watershed
    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=cfg.ws_min_distance, labels=mask, exclude_border=False)
    
    markers = np.zeros_like(mask, dtype=np.int32)
    for j, (r, c) in enumerate(coords, start=1):
        markers[r, c] = j
        
    labels = segmentation.watershed(-dist, markers, mask=mask)
    
    # Properly initializing the metadata_registry to fix the SyntaxError
    metadata_registry = []  
    patches = []
    
    half_p = cfg.patch_size // 2
    for lab in range(1, int(labels.max()) + 1):
        frag = (labels == lab).astype(np.uint8)
        cnts, _ = cv2.findContours(frag, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for cnt in cnts:
            if cv2.contourArea(cnt) < cfg.det_min_area:
                continue
            
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
                
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            # Boundary checks for patch extraction
            if cy - half_p < 0 or cy + half_p >= roi.shape[0] or cx - half_p < 0 or cx + half_p >= roi.shape[1]:
                continue
            
            patch = roi[cy - half_p : cy + half_p, cx - half_p : cx + half_p]
            patches.append(patch.astype(np.float32) / 255.0)
            
            metadata_registry.append({
                "center_x": cx + cfg.roi_x,
                "center_y": cy + cfg.roi_y,
                "area": float(cv2.contourArea(cnt)),
                "max_intensity": float(np.max(roi[frag.astype(bool)]))
            })
            
    return patches, pd.DataFrame(metadata_registry)

# ---------------------------------------------------------
# 4. Pipeline Execution: Autoencoder Training & GMM
# ---------------------------------------------------------
def run_unsupervised_pipeline(image_path: str):
    print("Initiating Smart Unsupervised Pipeline...")
    
    # Ensure reproducibility
    torch.manual_seed(CFG.random_state)
    np.random.seed(CFG.random_state)
    
    # 1. Extract Candidate Patches
    patches, metadata_df = isolate_track_patches(image_path, CFG)
    if len(patches) == 0:
        print("No valid tracks found.")
        return
        
    print(f"Extracted {len(patches)} valid track candidates. Training Autoencoder...")
    
    # Prepare PyTorch Tensors
    tensor_patches = torch.tensor(np.array(patches)).unsqueeze(1) # shape: (N, 1, 48, 48)
    dataset = TensorDataset(tensor_patches, tensor_patches)
    dataloader = DataLoader(dataset, batch_size=CFG.ae_batch, shuffle=True)
    
    # 2. Train Convolutional Autoencoder
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TrackAutoencoder(latent_dim=CFG.latent_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG.ae_lr)
    
    for epoch in range(CFG.ae_epochs):
        model.train()
        epoch_loss = 0
        for batch_data, _ in dataloader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            reconstructed, _ = model(batch_data)
            loss = criterion(reconstructed, batch_data)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
    # 3. Extract Deep Embeddings
    model.eval()
    with torch.no_grad():
        tensor_patches = tensor_patches.to(device)
        _, latent_embeddings = model(tensor_patches)
        latent_vectors = latent_embeddings.cpu().numpy()
        
    print(f"Latent embeddings extracted. Dimension: {latent_vectors.shape[1]}")
    
    # 4. Gaussian Mixture Model (GMM) Clustering
    print(f"Applying Gaussian Mixture Model with K={CFG.n_clusters} clusters...")
    scaler = StandardScaler()
    scaled_embeddings = scaler.fit_transform(latent_vectors)
    
    gmm = GaussianMixture(n_components=CFG.n_clusters, covariance_type='full', random_state=CFG.random_state)
    cluster_labels = gmm.fit_predict(scaled_embeddings)
    
    # 5. Append results to registry
    metadata_df["gmm_cluster"] = cluster_labels
    
    # Generate scatter plot mapping
    plt.figure(figsize=(9, 6))
    scatter = plt.scatter(latent_vectors[:, 0], latent_vectors[:, 1], c=cluster_labels, cmap='inferno', s=15, alpha=0.8)
    plt.colorbar(scatter, label="GMM Cluster ID")
    plt.title("Latent Space Embeddings (PCA representation) via GMM")
    plt.xlabel("Latent Dimension 1")
    plt.ylabel("Latent Dimension 2")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()
    
    return metadata_df

# Execute the algorithm on the target sample
df_results = run_unsupervised_pipeline("data_test/LBS255611.jpg")
if df_results is not None:
    print(df_results.head())


Initiating Smart Unsupervised Pipeline...


[ WARN:0@53.916] global loadsave.cpp:278 findDecoder imread_('./data_test/LBS255611.jpg'): can't open/read file: check file path/integrity
/var/folders/tf/18pztg5s60xb9b8nz75clkzc0000gn/T/ipykernel_22127/2274659906.py:106: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=cfg.det_min_area)


Extracted 1 valid track candidates. Training Autoencoder...
Latent embeddings extracted. Dimension: 32
Applying Gaussian Mixture Model with K=4 clusters...


ValueError: Found array with 1 sample(s) (shape=(1, 32)) while a minimum of 2 is required by GaussianMixture.